# Chameleon setup notebook for Smart Transaction Categorization
Use this notebook inside the Chameleon Jupyter environment. It creates a lease, a VM, the required security groups, and a floating IP for your CPU baseline and model-optimization work.

Before you run this notebook, delete any old lease/server with the same name in Horizon. The online evaluation lab uses a `KVM@TACC` `m1.medium` VM and attaches security groups before exposing FastAPI, Jupyter, Prometheus, and Grafana. Your project proposal also expects a single Chameleon VM to be sufficient for the lightweight CPU classifier. fileciteturn12file5 fileciteturn12file13

In [ ]:
from chi import server, context, lease, network
import chi, os, time, datetime, json

# Replace this with your real project suffix if needed.
PROJECT_SUFFIX = "proj08"

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
lease_name = f"lease-tx-serving-{username}-{PROJECT_SUFFIX}"
server_name = f"node-tx-serving-{username}-{PROJECT_SUFFIX}"
print({"lease_name": lease_name, "server_name": server_name})


## 1. Create a 6-hour lease for one `m1.medium` VM

In [ ]:
l = lease.Lease(lease_name, duration=datetime.timedelta(hours=6))
l.add_flavor_reservation(id=chi.server.get_flavor_id("m1.medium"), amount=1)
l.submit(idempotent=True)
l.show()


## 2. Launch the VM with `CC-Ubuntu24.04`

In [ ]:
s = server.Server(
    server_name,
    image_name="CC-Ubuntu24.04",
    flavor_name=l.get_reserved_flavors()[0].name
)
s.submit(idempotent=True)
s.refresh()
s.show()


## 3. Create and attach security groups

In [ ]:
security_groups = [
    {'name': f"allow-ssh-{PROJECT_SUFFIX}", 'port': 22, 'description': "Enable SSH traffic on TCP port 22"},
    {'name': f"allow-8000-{PROJECT_SUFFIX}", 'port': 8000, 'description': "Enable FastAPI traffic on TCP port 8000"},
    {'name': f"allow-8888-{PROJECT_SUFFIX}", 'port': 8888, 'description': "Enable Jupyter traffic on TCP port 8888"},
    {'name': f"allow-3000-{PROJECT_SUFFIX}", 'port': 3000, 'description': "Enable Grafana traffic on TCP port 3000"},
    {'name': f"allow-9090-{PROJECT_SUFFIX}", 'port': 9090, 'description': "Enable Prometheus traffic on TCP port 9090"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        'name': sg['name'],
        'description': sg['description'],
    })
    secgroup.add_rule(direction='ingress', protocol='tcp', port=sg['port'])
    secgroup.submit(idempotent=True)
    try:
        s.add_security_group(sg['name'])
    except Exception:
        pass

print("Attached security groups:", [sg['name'] for sg in security_groups])

## 4. Allocate and associate a floating IP

In [ ]:
s.associate_floating_ip()
time.sleep(5)
s.refresh()
s.check_connectivity()
s.show(type="widget")

## 5. Print the SSH command you will run from your Mac

In [ ]:
s.refresh()

addresses = getattr(s, "addresses", None)
print("addresses:", addresses)

floating_ip = None
if isinstance(addresses, dict):
    for net_name, addr_list in addresses.items():
        for addr in addr_list:
            if isinstance(addr, dict) and addr.get("OS-EXT-IPS:type") == "floating":
                floating_ip = addr.get("addr")
                break
        if floating_ip:
            break

if floating_ip:
    print(f"ssh -i ~/.ssh/id_rsa_chameleon cc@{floating_ip}")
    print(f"FastAPI docs: http://{floating_ip}:8000/docs")
    print(f"Jupyter:      http://{floating_ip}:8888")
    print(f"Prometheus:   http://{floating_ip}:9090")
    print(f"Grafana:      http://{floating_ip}:3000")
else:
    print("No floating IP found automatically. Please read it from s.show(type='widget').")
